# Packet P-034 — Out-of-Distribution Detection

**Decision question:** Can we flag new devices that fall outside the training distribution, so lab partners know when a prediction is extrapolation vs interpolation?

**Method:** Fit isolation forest on training data features, compute OOD scores for test set, partition into in-distribution and OOD halves, compare tau-b.

**Fastest falsifier:** If all 3 panel devices are in the top-10% most distant (OOD), the panel itself is in extrapolation territory.

**Success:** In-distribution half has tau-b > 0.20 and all 3 panel devices classified in-distribution.
**Kill:** No significant tau-b difference between halves, or panel devices are OOD.

In [1]:
"""Cell 1 — Load data, setup."""
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor, IsolationForest
from sklearn.model_selection import train_test_split
from scipy.stats import kendalltau
from scipy.spatial.distance import mahalanobis
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('perovskite_stability_clean.csv')
FEATURES = [
    'Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
    'MA', 'FA', 'Cs',
    'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured',
    'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF'
]
TARGET = 'Stability_PCE_T80'
ET_PARAMS = dict(n_estimators=700, max_features='sqrt', min_samples_split=3,
                 min_samples_leaf=1, bootstrap=False, random_state=42)
PANEL = [850, 1320, 119]

X = df[FEATURES].fillna(0)
y = np.log1p(df[TARGET])
print(f"Dataset: {len(X)} devices, {len(FEATURES)} features")

Dataset: 1543 devices, 16 features


In [2]:
"""Cell 2 — OOD detection via Isolation Forest + Mahalanobis across 10 splits."""

N_SPLITS = 10

# Results storage
id_taus = []
ood_taus = []
panel_ood_flags = {d: [] for d in PANEL}

for seed in range(N_SPLITS):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)
    
    # Train ET model
    model = ExtraTreesRegressor(**ET_PARAMS)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    
    # Fit Isolation Forest on training data
    iso = IsolationForest(n_estimators=300, contamination=0.1, random_state=seed)
    iso.fit(X_tr)
    
    # Score test set (lower = more anomalous)
    ood_scores = iso.decision_function(X_te)
    
    # Split test set at median OOD score
    median_score = np.median(ood_scores)
    in_dist_mask = ood_scores >= median_score
    ood_mask = ood_scores < median_score
    
    # Tau-b for each half
    if in_dist_mask.sum() >= 10 and ood_mask.sum() >= 10:
        tau_id, _ = kendalltau(y_te.values[in_dist_mask], preds[in_dist_mask])
        tau_ood, _ = kendalltau(y_te.values[ood_mask], preds[ood_mask])
        id_taus.append(tau_id)
        ood_taus.append(tau_ood)
    
    # Check panel devices
    test_idx = X_te.index.tolist()
    for d in PANEL:
        if d in test_idx:
            pos = test_idx.index(d)
            is_ood = ood_scores[pos] < median_score
            panel_ood_flags[d].append(is_ood)

    if (seed + 1) % 5 == 0:
        print(f"  Completed {seed+1}/{N_SPLITS} splits")

# Also compute Mahalanobis distance for panel devices on full dataset
# Use regularized covariance to handle near-singular matrices
from numpy.linalg import pinv

cov_matrix = X.cov().values
# Regularize
cov_reg = cov_matrix + np.eye(len(FEATURES)) * 1e-4
cov_inv = pinv(cov_reg)
mean_vec = X.mean().values

print(f"\n{'=' * 70}")
print("OOD DETECTION RESULTS")
print(f"{'=' * 70}")

print(f"\nTau-b comparison (mean over {len(id_taus)} splits):")
print(f"  In-distribution half: {np.mean(id_taus):.4f} ± {np.std(id_taus):.4f}")
print(f"  OOD half:             {np.mean(ood_taus):.4f} ± {np.std(ood_taus):.4f}")
print(f"  Delta (ID - OOD):     {np.mean(id_taus) - np.mean(ood_taus):+.4f}")

# Panel device analysis
print(f"\n── Panel Device OOD Status ──")
for d in PANEL:
    if panel_ood_flags[d]:
        ood_rate = np.mean(panel_ood_flags[d])
        # Mahalanobis distance
        x_d = X.loc[d].values
        m_dist = mahalanobis(x_d, mean_vec, cov_inv)
        # Percentile of Mahalanobis distance
        all_dists = [mahalanobis(X.iloc[i].values, mean_vec, cov_inv) for i in range(len(X))]
        pctile = np.mean(np.array(all_dists) <= m_dist) * 100
        status = "OOD" if ood_rate > 0.5 else "In-distribution"
        print(f"  Device {d}: {status} (OOD rate {ood_rate:.0%}, Mahal dist pctile {pctile:.0f}%)")
    else:
        print(f"  Device {d}: never in test set")

  Completed 5/10 splits


  Completed 10/10 splits

OOD DETECTION RESULTS

Tau-b comparison (mean over 10 splits):
  In-distribution half: 0.2493 ± 0.0473
  OOD half:             0.3384 ± 0.0548
  Delta (ID - OOD):     -0.0891

── Panel Device OOD Status ──
  Device 850: OOD (OOD rate 100%, Mahal dist pctile 83%)
  Device 1320: OOD (OOD rate 100%, Mahal dist pctile 80%)
  Device 119: OOD (OOD rate 100%, Mahal dist pctile 73%)


In [3]:
"""Cell 3 — Evaluate thresholds and save."""

# Success: ID half tau-b > 0.20 AND all panel devices in-distribution
# Kill: no significant tau-b difference OR panel devices are OOD

mean_id_tau = np.mean(id_taus)
mean_ood_tau = np.mean(ood_taus)
delta_tau = mean_id_tau - mean_ood_tau

# Panel check
panel_all_id = True
panel_status = {}
for d in PANEL:
    if panel_ood_flags[d]:
        ood_rate = np.mean(panel_ood_flags[d])
        panel_status[d] = ood_rate
        if ood_rate > 0.5:
            panel_all_id = False
    else:
        panel_status[d] = float('nan')

print("── Evaluation ──\n")
print(f"ID half mean tau-b:  {mean_id_tau:.4f}")
print(f"OOD half mean tau-b: {mean_ood_tau:.4f}")
print(f"Delta:               {delta_tau:+.4f}")
print(f"Panel all in-dist:   {'YES' if panel_all_id else 'NO'}")

# Discriminative power: is the delta meaningful?
significant_delta = abs(delta_tau) > 0.03

if mean_id_tau > 0.20 and panel_all_id and significant_delta:
    status = "Confirmed"
elif significant_delta:
    status = "Promising"
else:
    status = "Negative"

# Save
save_rows = [
    {'metric': 'mean_tau_in_distribution', 'value': mean_id_tau},
    {'metric': 'mean_tau_ood', 'value': mean_ood_tau},
    {'metric': 'delta_tau', 'value': delta_tau},
    {'metric': 'panel_all_in_distribution', 'value': panel_all_id},
]
for d in PANEL:
    save_rows.append({'metric': f'panel_{d}_ood_rate', 'value': panel_status[d]})

pd.DataFrame(save_rows).to_csv('Packet_P034_OOD_Detection.csv', index=False)
print(f"\nSaved: Packet_P034_OOD_Detection.csv")

print(f"\n{'=' * 60}")
print(f"P-034 STATUS: {status}")
print(f"{'=' * 60}")
if status == "Confirmed":
    print("OOD detector discriminates prediction quality. Panel devices are in-distribution.")
    print("Can ship OOD flag alongside predictions for partner trust calibration.")
elif status == "Promising":
    print("OOD detector shows some discrimination but panel placement is mixed.")
else:
    print("OOD detector does not meaningfully separate prediction quality.")
    print("Feature-space distance is not a useful trust signal for this model.")

── Evaluation ──

ID half mean tau-b:  0.2493
OOD half mean tau-b: 0.3384
Delta:               -0.0891
Panel all in-dist:   NO

Saved: Packet_P034_OOD_Detection.csv

P-034 STATUS: Promising
OOD detector shows some discrimination but panel placement is mixed.
